# Metricas de Evaluacion para Sistemas RAG

## Objetivo
Implementar y comprender las metricas clave para evaluar sistemas de Retrieval-Augmented Generation (RAG).

## Por que importa la evaluacion?

Un sistema RAG tiene multiples componentes que pueden fallar de diferentes maneras:

- **El retriever** puede traer contexto irrelevante
- **El generador** puede ignorar el contexto y alucinar
- **La respuesta** puede ser correcta pero no responder la pregunta

Sin metricas de evaluacion, no tenemos forma objetiva de saber si nuestro sistema funciona
bien o de medir el impacto de los cambios que hacemos.

### Metricas que implementaremos

| Metrica | Que mide | Pregunta clave |
|---------|----------|----------------|
| **Faithfulness** | Fidelidad al contexto | La respuesta esta respaldada por el contexto? |
| **Answer Relevancy** | Relevancia de la respuesta | La respuesta contesta la pregunta? |
| **Context Precision** | Precision del contexto | El contexto recuperado es relevante? |

## Contenido
1. Dataset de Evaluacion
2. Faithfulness (Fidelidad)
3. Answer Relevancy (Relevancia)
4. Context Precision
5. Evaluacion End-to-End
6. Ejercicio

## Configuracion del Entorno

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

# Verificar credenciales
if not os.getenv("GITHUB_TOKEN"):
    raise EnvironmentError("GITHUB_TOKEN no configurado. Copia .env.example a .env y agrega tu token.")
print("Entorno configurado correctamente")

Entorno configurado correctamente


In [2]:
from openai import OpenAI
import json

client = OpenAI(
    base_url=os.getenv("GITHUB_BASE_URL", "https://models.inference.ai.azure.com"),
    api_key=os.getenv("GITHUB_TOKEN")
)

MODELO = "gpt-4o-mini"
print(f"Cliente configurado con modelo: {MODELO}")

Cliente configurado con modelo: gpt-4o-mini


In [3]:
def llamar_llm(prompt, system_prompt="Eres un asistente de evaluacion.", temperatura=0.0, max_tokens=500):
    """Realiza una llamada al LLM y retorna la respuesta como texto."""
    respuesta = client.chat.completions.create(
        model=MODELO,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=temperatura,
        max_tokens=max_tokens
    )
    return respuesta.choices[0].message.content

---
## Seccion 1: Dataset de Evaluacion

Para evaluar un sistema RAG necesitamos un dataset con:
- **Pregunta**: Lo que el usuario consulta
- **Contexto**: Los documentos recuperados por el retriever
- **Respuesta generada**: Lo que el sistema RAG produce
- **Respuesta esperada** (ground truth): La respuesta correcta de referencia

Crearemos un dataset pequeno pero representativo sobre una tematica de tecnologia.

In [4]:
# Dataset de evaluacion: preguntas sobre Python y ciencia de datos
dataset_evaluacion = [
    {
        "id": 1,
        "pregunta": "Que es NumPy y para que se utiliza?",
        "contexto": "NumPy es una biblioteca fundamental para computacion cientifica en Python. Proporciona soporte para arrays multidimensionales y matrices, junto con una coleccion de funciones matematicas de alto nivel para operar con estos arrays. NumPy es la base de muchas otras bibliotecas como pandas, scikit-learn y matplotlib.",
        "respuesta_generada": "NumPy es una biblioteca de Python para computacion cientifica. Se utiliza principalmente para trabajar con arrays multidimensionales y realizar operaciones matematicas eficientes. Es la base de otras bibliotecas como pandas y scikit-learn.",
        "respuesta_esperada": "NumPy es una biblioteca de Python para computacion cientifica que proporciona soporte para arrays multidimensionales y funciones matematicas de alto nivel."
    },
    {
        "id": 2,
        "pregunta": "Como funciona el algoritmo de Random Forest?",
        "contexto": "Random Forest es un algoritmo de aprendizaje supervisado que crea multiples arboles de decision durante el entrenamiento. Cada arbol se entrena con una muestra aleatoria del dataset (bootstrap) y en cada nodo se considera un subconjunto aleatorio de features. La prediccion final se obtiene por votacion mayoritaria (clasificacion) o promedio (regresion).",
        "respuesta_generada": "Random Forest funciona creando muchos arboles de decision. Cada arbol vota y la respuesta mas votada gana. Es un algoritmo de deep learning muy popular.",
        "respuesta_esperada": "Random Forest crea multiples arboles de decision entrenados con muestras bootstrap y subconjuntos aleatorios de features. La prediccion se hace por votacion mayoritaria o promedio."
    },
    {
        "id": 3,
        "pregunta": "Que es Docker y cuales son sus ventajas?",
        "contexto": "Docker es una plataforma de contenedorizacion que permite empaquetar aplicaciones y sus dependencias en contenedores ligeros y portables. Los contenedores comparten el kernel del sistema operativo host, lo que los hace mas eficientes que las maquinas virtuales. Docker facilita el despliegue consistente en diferentes entornos.",
        "respuesta_generada": "Docker es una plataforma que permite crear contenedores para empaquetar aplicaciones con sus dependencias. Sus ventajas incluyen portabilidad, eficiencia al compartir el kernel del OS, y despliegue consistente entre entornos de desarrollo y produccion.",
        "respuesta_esperada": "Docker es una plataforma de contenedorizacion que empaqueta aplicaciones con dependencias en contenedores ligeros. Ventajas: portabilidad, eficiencia y despliegue consistente."
    },
    {
        "id": 4,
        "pregunta": "Que es una API REST?",
        "contexto": "REST (Representational State Transfer) es un estilo arquitectonico para disenar servicios web. Una API REST utiliza metodos HTTP estandar (GET, POST, PUT, DELETE) para realizar operaciones CRUD sobre recursos. Los recursos se identifican mediante URLs y las respuestas generalmente se devuelven en formato JSON.",
        "respuesta_generada": "Una API REST es una interfaz que permite la comunicacion entre sistemas usando el protocolo HTTP. Utiliza metodos como GET y POST para manipular recursos identificados por URLs, con respuestas en formato JSON.",
        "respuesta_esperada": "Una API REST es un servicio web que sigue el estilo arquitectonico REST, usando metodos HTTP para operaciones CRUD sobre recursos identificados por URLs."
    },
    {
        "id": 5,
        "pregunta": "Que es el overfitting en machine learning?",
        "contexto": "El overfitting ocurre cuando un modelo de machine learning se ajusta demasiado a los datos de entrenamiento, capturando ruido y patrones espurios. Un modelo con overfitting tiene un excelente rendimiento en datos de entrenamiento pero un rendimiento pobre en datos nuevos (test). Tecnicas como regularizacion, cross-validation y dropout ayudan a prevenirlo.",
        "respuesta_generada": "El overfitting es cuando un modelo se memoriza los datos de entrenamiento en lugar de aprender patrones generales. Esto causa mal rendimiento con datos nuevos. Se puede prevenir con regularizacion y cross-validation.",
        "respuesta_esperada": "El overfitting es cuando un modelo se ajusta demasiado a los datos de entrenamiento, capturando ruido. Tiene buen rendimiento en entrenamiento pero malo en datos nuevos."
    },
    {
        "id": 6,
        "pregunta": "Cuales son las diferencias entre SQL y NoSQL?",
        "contexto": "Las bases de datos SQL (relacionales) organizan datos en tablas con esquemas fijos y usan el lenguaje SQL para consultas. Ejemplos: PostgreSQL, MySQL. Las bases NoSQL no requieren esquemas fijos y pueden ser de tipo documento (MongoDB), clave-valor (Redis), columnar (Cassandra) o grafo (Neo4j). SQL garantiza ACID, mientras NoSQL prioriza escalabilidad horizontal.",
        "respuesta_generada": "SQL usa tablas con esquemas fijos y el lenguaje SQL. NoSQL es mas flexible, no requiere esquemas fijos y viene en varios tipos como documento y clave-valor. SQL garantiza transacciones ACID mientras NoSQL prioriza la escalabilidad.",
        "respuesta_esperada": "SQL usa tablas con esquemas fijos y garantiza ACID. NoSQL es flexible, sin esquemas fijos, con tipos como documento, clave-valor, columnar y grafo, priorizando escalabilidad."
    },
    {
        "id": 7,
        "pregunta": "Que es Kubernetes?",
        "contexto": "Kubernetes (K8s) es una plataforma de orquestacion de contenedores de codigo abierto desarrollada originalmente por Google. Automatiza el despliegue, escalado y gestion de aplicaciones en contenedores. Sus componentes principales incluyen pods, services, deployments y namespaces.",
        "respuesta_generada": "Kubernetes es una herramienta para gestionar servidores fisicos en la nube. Fue creado por Amazon y se usa principalmente para administrar bases de datos.",
        "respuesta_esperada": "Kubernetes es una plataforma de orquestacion de contenedores de codigo abierto de Google que automatiza despliegue, escalado y gestion de aplicaciones containerizadas."
    }
]

print(f"Dataset creado con {len(dataset_evaluacion)} ejemplos")
print("\nEjemplo del dataset:")
ejemplo = dataset_evaluacion[0]
print(f"  Pregunta: {ejemplo['pregunta']}")
print(f"  Contexto: {ejemplo['contexto'][:80]}...")
print(f"  Respuesta generada: {ejemplo['respuesta_generada'][:80]}...")

Dataset creado con 7 ejemplos

Ejemplo del dataset:
  Pregunta: Que es NumPy y para que se utiliza?
  Contexto: NumPy es una biblioteca fundamental para computacion cientifica en Python. Propo...
  Respuesta generada: NumPy es una biblioteca de Python para computacion cientifica. Se utiliza princi...


---
## Seccion 2: Faithfulness (Fidelidad)

La **fidelidad** mide si la respuesta generada esta respaldada por el contexto proporcionado.
Una respuesta con alta fidelidad no contiene informacion que no este en el contexto
(es decir, no "alucina").

### Metodo
1. Descomponer la respuesta en afirmaciones individuales (claims)
2. Verificar si cada afirmacion esta respaldada por el contexto
3. Calcular: `faithfulness = afirmaciones_respaldadas / total_afirmaciones`

In [5]:
def extraer_afirmaciones(respuesta):
    """Descompone una respuesta en afirmaciones individuales verificables."""
    prompt = f"""Descompone la siguiente respuesta en afirmaciones individuales y verificables.
Cada afirmacion debe ser una oracion simple que pueda ser verdadera o falsa.

RESPUESTA:
{respuesta}

Responde UNICAMENTE con un JSON valido:
{{"afirmaciones": ["afirmacion 1", "afirmacion 2", ...]}}"""
    
    resultado = llamar_llm(prompt)
    texto_limpio = resultado.strip()
    if texto_limpio.startswith("```"):
        lineas = texto_limpio.split("\n")
        texto_limpio = "\n".join(lineas[1:-1])
    
    try:
        datos = json.loads(texto_limpio)
        return datos.get("afirmaciones", [])
    except json.JSONDecodeError:
        # Fallback: dividir por oraciones
        return [s.strip() for s in respuesta.split(".") if s.strip()]


def verificar_afirmacion(afirmacion, contexto):
    """Verifica si una afirmacion esta respaldada por el contexto."""
    prompt = f"""Determina si la siguiente afirmacion esta respaldada por el contexto dado.

CONTEXTO:
{contexto}

AFIRMACION:
{afirmacion}

Responde UNICAMENTE con un JSON valido:
{{"respaldada": true/false, "explicacion": "breve explicacion"}}"""
    
    resultado = llamar_llm(prompt)
    texto_limpio = resultado.strip()
    if texto_limpio.startswith("```"):
        lineas = texto_limpio.split("\n")
        texto_limpio = "\n".join(lineas[1:-1])
    
    try:
        datos = json.loads(texto_limpio)
        return datos
    except json.JSONDecodeError:
        return {"respaldada": False, "explicacion": "Error al parsear respuesta"}


def calcular_faithfulness(respuesta, contexto):
    """Calcula el puntaje de faithfulness de una respuesta dado un contexto.
    
    Returns:
        dict con puntaje (0-1), afirmaciones y detalles de verificacion
    """
    # Paso 1: Extraer afirmaciones
    afirmaciones = extraer_afirmaciones(respuesta)
    
    if not afirmaciones:
        return {"puntaje": 0.0, "afirmaciones": [], "detalle": []}
    
    # Paso 2: Verificar cada afirmacion
    detalle = []
    respaldadas = 0
    
    for afirmacion in afirmaciones:
        verificacion = verificar_afirmacion(afirmacion, contexto)
        detalle.append({
            "afirmacion": afirmacion,
            "respaldada": verificacion.get("respaldada", False),
            "explicacion": verificacion.get("explicacion", "")
        })
        if verificacion.get("respaldada", False):
            respaldadas += 1
    
    # Paso 3: Calcular puntaje
    puntaje = respaldadas / len(afirmaciones)
    
    return {
        "puntaje": puntaje,
        "afirmaciones_total": len(afirmaciones),
        "afirmaciones_respaldadas": respaldadas,
        "detalle": detalle
    }

In [6]:
# Evaluar faithfulness en algunos ejemplos del dataset
print("=" * 60)
print("EVALUACION DE FAITHFULNESS")
print("=" * 60)

resultados_faithfulness = []

for item in dataset_evaluacion[:4]:  # Evaluar los primeros 4 para ahorrar tokens
    print(f"\n--- Ejemplo {item['id']}: {item['pregunta']} ---")
    resultado = calcular_faithfulness(item["respuesta_generada"], item["contexto"])
    resultados_faithfulness.append({"id": item["id"], **resultado})
    
    print(f"Puntaje: {resultado['puntaje']:.2f} ({resultado['afirmaciones_respaldadas']}/{resultado['afirmaciones_total']})")
    for det in resultado["detalle"]:
        estado = "OK" if det["respaldada"] else "NO"
        print(f"  [{estado}] {det['afirmacion'][:70]}")
        if not det["respaldada"]:
            print(f"       Razon: {det['explicacion'][:60]}")

EVALUACION DE FAITHFULNESS

--- Ejemplo 1: Que es NumPy y para que se utiliza? ---
Puntaje: 1.00 (6/6)
  [OK] NumPy es una biblioteca de Python.
  [OK] NumPy se utiliza para computación científica.
  [OK] NumPy se utiliza principalmente para trabajar con arrays multidimensio
  [OK] NumPy se utiliza para realizar operaciones matemáticas eficientes.
  [OK] NumPy es la base de otras bibliotecas como pandas.
  [OK] NumPy es la base de otras bibliotecas como scikit-learn.

--- Ejemplo 2: Como funciona el algoritmo de Random Forest? ---
Puntaje: 0.50 (2/4)
  [OK] Random Forest funciona creando muchos árboles de decisión.
  [OK] Cada árbol vota y la respuesta más votada gana.
  [NO] Random Forest es un algoritmo de deep learning.
       Razon: Random Forest es un algoritmo de aprendizaje supervisado, pe
  [NO] Random Forest es un algoritmo muy popular.
       Razon: El contexto no menciona la popularidad de Random Forest, sol

--- Ejemplo 3: Que es Docker y cuales son sus ventajas? ---
Puntaj

---
## Seccion 3: Answer Relevancy (Relevancia de la Respuesta)

La **relevancia** mide si la respuesta realmente contesta la pregunta del usuario.
Una respuesta puede ser fiel al contexto pero no relevante si no aborda la pregunta.

### Metodo
Usamos un enfoque basado en el LLM como juez: le pedimos que evalue
que tan bien la respuesta aborda la pregunta, en una escala de 0 a 1.

In [7]:
def calcular_relevancia(pregunta, respuesta):
    """Calcula la relevancia de una respuesta respecto a la pregunta.
    
    Usa el LLM como juez para evaluar si la respuesta aborda la pregunta.
    
    Returns:
        dict con puntaje (0-1) y justificacion
    """
    prompt = f"""Evalua que tan relevante es la siguiente respuesta para la pregunta dada.
Una respuesta relevante aborda directamente lo que se pregunta, es completa y no incluye
informacion innecesaria.

PREGUNTA: {pregunta}

RESPUESTA: {respuesta}

Evalua en una escala de 0.0 a 1.0 donde:
- 0.0 = Completamente irrelevante, no aborda la pregunta
- 0.5 = Parcialmente relevante, aborda algunos aspectos
- 1.0 = Completamente relevante, aborda todos los aspectos de la pregunta

Responde UNICAMENTE con un JSON valido:
{{"puntaje": <0.0 a 1.0>, "justificacion": "breve explicacion"}}"""
    
    resultado = llamar_llm(prompt)
    texto_limpio = resultado.strip()
    if texto_limpio.startswith("```"):
        lineas = texto_limpio.split("\n")
        texto_limpio = "\n".join(lineas[1:-1])
    
    try:
        datos = json.loads(texto_limpio)
        return datos
    except json.JSONDecodeError:
        return {"puntaje": 0.0, "justificacion": f"Error al parsear: {resultado}"}

In [8]:
# Evaluar relevancia en el dataset
print("=" * 60)
print("EVALUACION DE ANSWER RELEVANCY")
print("=" * 60)

resultados_relevancia = []

for item in dataset_evaluacion:
    resultado = calcular_relevancia(item["pregunta"], item["respuesta_generada"])
    resultados_relevancia.append({"id": item["id"], **resultado})
    
    print(f"\nEjemplo {item['id']}: {item['pregunta']}")
    print(f"  Puntaje: {resultado['puntaje']:.2f}")
    print(f"  Justificacion: {resultado['justificacion']}")

# Promedio
promedio_relevancia = sum(r["puntaje"] for r in resultados_relevancia) / len(resultados_relevancia)
print(f"\nPromedio de relevancia: {promedio_relevancia:.2f}")

EVALUACION DE ANSWER RELEVANCY

Ejemplo 1: Que es NumPy y para que se utiliza?
  Puntaje: 1.00
  Justificacion: La respuesta define claramente qué es NumPy y menciona su uso principal, así como su relación con otras bibliotecas, abordando completamente la pregunta.

Ejemplo 2: Como funciona el algoritmo de Random Forest?
  Puntaje: 0.50
  Justificacion: La respuesta menciona que Random Forest crea muchos árboles de decisión y que cada árbol vota, lo cual es relevante. Sin embargo, se considera parcialmente relevante porque no explica cómo se construyen esos árboles ni cómo se realiza la votación. Además, clasificarlo como un algoritmo de deep learning es incorrecto, ya que Random Forest es un algoritmo de aprendizaje de conjunto, no de deep learning.

Ejemplo 3: Que es Docker y cuales son sus ventajas?
  Puntaje: 1.00
  Justificacion: La respuesta define claramente qué es Docker y menciona varias ventajas relevantes, abordando completamente la pregunta.

Ejemplo 4: Que es una API REST?

---
## Seccion 4: Context Precision (Precision del Contexto)

La **precision del contexto** mide si el contexto recuperado por el retriever
es realmente relevante para responder la pregunta.

Un contexto con alta precision contiene informacion directamente util para
responder la pregunta, sin incluir informacion irrelevante.

### Metodo
Evaluamos si el contexto contiene la informacion necesaria para responder
la pregunta comparandolo con la respuesta esperada (ground truth).

In [9]:
def calcular_context_precision(pregunta, contexto, respuesta_esperada):
    """Evalua si el contexto recuperado es relevante para responder la pregunta.
    
    Args:
        pregunta: La pregunta del usuario
        contexto: El contexto recuperado por el retriever
        respuesta_esperada: La respuesta correcta (ground truth)
    
    Returns:
        dict con puntaje (0-1), relevancia y cobertura
    """
    prompt = f"""Evalua la calidad del contexto recuperado para responder la pregunta.

PREGUNTA: {pregunta}

CONTEXTO RECUPERADO:
{contexto}

RESPUESTA CORRECTA (referencia):
{respuesta_esperada}

Evalua dos aspectos:
1. RELEVANCIA: El contexto es relevante para la pregunta? (0.0 a 1.0)
2. COBERTURA: El contexto contiene suficiente informacion para generar la respuesta correcta? (0.0 a 1.0)

Responde UNICAMENTE con un JSON valido:
{{
    "relevancia": <0.0 a 1.0>,
    "cobertura": <0.0 a 1.0>,
    "puntaje_final": <promedio de relevancia y cobertura>,
    "justificacion": "breve explicacion"
}}"""
    
    resultado = llamar_llm(prompt)
    texto_limpio = resultado.strip()
    if texto_limpio.startswith("```"):
        lineas = texto_limpio.split("\n")
        texto_limpio = "\n".join(lineas[1:-1])
    
    try:
        datos = json.loads(texto_limpio)
        return datos
    except json.JSONDecodeError:
        return {
            "relevancia": 0.0,
            "cobertura": 0.0,
            "puntaje_final": 0.0,
            "justificacion": f"Error al parsear: {resultado}"
        }

In [10]:
# Evaluar context precision en el dataset
print("=" * 60)
print("EVALUACION DE CONTEXT PRECISION")
print("=" * 60)

resultados_contexto = []

for item in dataset_evaluacion:
    resultado = calcular_context_precision(
        item["pregunta"],
        item["contexto"],
        item["respuesta_esperada"]
    )
    resultados_contexto.append({"id": item["id"], **resultado})
    
    print(f"\nEjemplo {item['id']}: {item['pregunta']}")
    print(f"  Relevancia: {resultado.get('relevancia', 0):.2f}")
    print(f"  Cobertura: {resultado.get('cobertura', 0):.2f}")
    print(f"  Puntaje final: {resultado.get('puntaje_final', 0):.2f}")
    print(f"  Justificacion: {resultado.get('justificacion', 'N/A')}")

# Promedios
prom_relevancia = sum(r.get("relevancia", 0) for r in resultados_contexto) / len(resultados_contexto)
prom_cobertura = sum(r.get("cobertura", 0) for r in resultados_contexto) / len(resultados_contexto)
prom_final = sum(r.get("puntaje_final", 0) for r in resultados_contexto) / len(resultados_contexto)

print(f"\nPromedios:")
print(f"  Relevancia: {prom_relevancia:.2f}")
print(f"  Cobertura: {prom_cobertura:.2f}")
print(f"  Puntaje final: {prom_final:.2f}")

EVALUACION DE CONTEXT PRECISION

Ejemplo 1: Que es NumPy y para que se utiliza?
  Relevancia: 1.00
  Cobertura: 1.00
  Puntaje final: 1.00
  Justificacion: El contexto recuperado es completamente relevante para la pregunta, ya que define claramente qué es NumPy y menciona su uso en computación científica, además de proporcionar información adicional sobre su relación con otras bibliotecas, lo que enriquece la respuesta.

Ejemplo 2: Como funciona el algoritmo de Random Forest?
  Relevancia: 1.00
  Cobertura: 1.00
  Puntaje final: 1.00
  Justificacion: El contexto es completamente relevante para la pregunta, ya que describe de manera precisa cómo funciona el algoritmo de Random Forest, incluyendo la creación de árboles de decisión, el uso de muestras bootstrap y la forma de hacer predicciones. Además, proporciona toda la información necesaria para generar la respuesta correcta.

Ejemplo 3: Que es Docker y cuales son sus ventajas?
  Relevancia: 1.00
  Cobertura: 0.80
  Puntaje final: 0.90

---
## Seccion 5: Evaluacion End-to-End

Ahora ejecutamos todas las metricas sobre el dataset completo y generamos
un reporte consolidado usando pandas.

Esto nos da una vision integral de la calidad del sistema RAG.

In [11]:
import pandas as pd

def evaluacion_completa(dataset):
    """Ejecuta todas las metricas de evaluacion sobre el dataset.
    
    Returns:
        DataFrame con los resultados de todas las metricas por ejemplo
    """
    resultados = []
    
    for item in dataset:
        print(f"Evaluando ejemplo {item['id']}: {item['pregunta'][:40]}...")
        
        # Faithfulness
        faith = calcular_faithfulness(item["respuesta_generada"], item["contexto"])
        
        # Relevancia
        relev = calcular_relevancia(item["pregunta"], item["respuesta_generada"])
        
        # Context Precision
        ctx = calcular_context_precision(
            item["pregunta"],
            item["contexto"],
            item["respuesta_esperada"]
        )
        
        resultados.append({
            "ID": item["id"],
            "Pregunta": item["pregunta"][:50],
            "Faithfulness": round(faith["puntaje"], 2),
            "Relevancia": round(relev.get("puntaje", 0), 2),
            "Context Precision": round(ctx.get("puntaje_final", 0), 2)
        })
    
    return pd.DataFrame(resultados)

In [12]:
# Ejecutar evaluacion completa (usamos un subconjunto para ahorrar tokens)
print("Ejecutando evaluacion end-to-end...\n")
df_resultados = evaluacion_completa(dataset_evaluacion[:5])

print("\n" + "=" * 70)
print("REPORTE DE EVALUACION END-TO-END")
print("=" * 70)
print()
print(df_resultados.to_string(index=False))

Ejecutando evaluacion end-to-end...

Evaluando ejemplo 1: Que es NumPy y para que se utiliza?...
Evaluando ejemplo 2: Como funciona el algoritmo de Random For...
Evaluando ejemplo 3: Que es Docker y cuales son sus ventajas?...
Evaluando ejemplo 4: Que es una API REST?...
Evaluando ejemplo 5: Que es el overfitting en machine learnin...

REPORTE DE EVALUACION END-TO-END

 ID                                     Pregunta  Faithfulness  Relevancia  Context Precision
  1          Que es NumPy y para que se utiliza?           1.0         1.0                1.0
  2 Como funciona el algoritmo de Random Forest?           0.5         0.5                1.0
  3     Que es Docker y cuales son sus ventajas?           1.0         1.0                0.9
  4                         Que es una API REST?           1.0         1.0                1.0
  5   Que es el overfitting en machine learning?           1.0         1.0                1.0


In [13]:
# Resumen estadistico
print("\n" + "=" * 50)
print("RESUMEN ESTADISTICO")
print("=" * 50)

metricas = ["Faithfulness", "Relevancia", "Context Precision"]
for metrica in metricas:
    valores = df_resultados[metrica]
    print(f"\n{metrica}:")
    print(f"  Promedio: {valores.mean():.2f}")
    print(f"  Minimo:   {valores.min():.2f}")
    print(f"  Maximo:   {valores.max():.2f}")
    print(f"  Std Dev:  {valores.std():.2f}")

# Puntaje global
puntaje_global = df_resultados[metricas].mean().mean()
print(f"\nPuntaje global del sistema RAG: {puntaje_global:.2f}")


RESUMEN ESTADISTICO

Faithfulness:
  Promedio: 0.90
  Minimo:   0.50
  Maximo:   1.00
  Std Dev:  0.22

Relevancia:
  Promedio: 0.90
  Minimo:   0.50
  Maximo:   1.00
  Std Dev:  0.22

Context Precision:
  Promedio: 0.98
  Minimo:   0.90
  Maximo:   1.00
  Std Dev:  0.04

Puntaje global del sistema RAG: 0.93


In [14]:
# Identificar ejemplos problematicos
print("\n" + "=" * 50)
print("EJEMPLOS PROBLEMATICOS")
print("=" * 50)

umbral = 0.7
problematicos = []

for _, row in df_resultados.iterrows():
    problemas = []
    for metrica in metricas:
        if row[metrica] < umbral:
            problemas.append(f"{metrica}={row[metrica]:.2f}")
    
    if problemas:
        problematicos.append(row)
        print(f"\nEjemplo {int(row['ID'])}: {row['Pregunta']}")
        print(f"  Problemas: {', '.join(problemas)}")

if not problematicos:
    print("\nNo se encontraron ejemplos con metricas por debajo del umbral ({umbral}).")
else:
    print(f"\nTotal de ejemplos problematicos: {len(problematicos)} de {len(df_resultados)}")


EJEMPLOS PROBLEMATICOS

Ejemplo 2: Como funciona el algoritmo de Random Forest?
  Problemas: Faithfulness=0.50, Relevancia=0.50

Total de ejemplos problematicos: 1 de 5


---
## Seccion 6: Ejercicio

### Tarea: Crear tu propio dataset de evaluacion y ejecutar las metricas

**Instrucciones:**
1. Elige un dominio de conocimiento (ej: medicina, derecho, cocina, deportes)
2. Crea un dataset de al menos 5 ejemplos con:
   - Pregunta
   - Contexto (simulando lo que un retriever devolveria)
   - Respuesta generada (simula respuestas con diferentes niveles de calidad)
   - Respuesta esperada (ground truth)
3. Incluye intencionalmente al menos:
   - 1 ejemplo con baja fidelidad (respuesta con informacion no presente en el contexto)
   - 1 ejemplo con baja relevancia (respuesta que no contesta la pregunta)
   - 1 ejemplo con contexto irrelevante
4. Ejecuta las metricas y analiza los resultados

In [15]:
# === ESPACIO PARA EL ESTUDIANTE ===
# Crea tu dataset de evaluacion aqui

mi_dataset = [
    {
        "id": 1,
        "pregunta": "[TU PREGUNTA AQUI]",
        "contexto": "[EL CONTEXTO RECUPERADO]",
        "respuesta_generada": "[LA RESPUESTA DEL SISTEMA RAG]",
        "respuesta_esperada": "[LA RESPUESTA CORRECTA]"
    },
    # Agrega al menos 4 ejemplos mas...
]

print(f"Dataset creado con {len(mi_dataset)} ejemplos")

Dataset creado con 1 ejemplos


In [16]:
# === ESPACIO PARA EL ESTUDIANTE ===
# Ejecuta la evaluacion completa sobre tu dataset

# Descomenta cuando tengas tu dataset listo:
# print("Ejecutando evaluacion sobre mi dataset...\n")
# mi_df_resultados = evaluacion_completa(mi_dataset)
# print("\n" + "=" * 70)
# print("MI REPORTE DE EVALUACION")
# print("=" * 70)
# print(mi_df_resultados.to_string(index=False))
# 
# # Resumen
# for metrica in ["Faithfulness", "Relevancia", "Context Precision"]:
#     print(f"\n{metrica}: promedio = {mi_df_resultados[metrica].mean():.2f}")

### Preguntas de reflexion

1. **Cual metrica fue la mas baja en tu dataset? Por que crees que sucede?**
   - (Tu respuesta aqui)

2. **Que estrategias usarias para mejorar la fidelidad (faithfulness) de un sistema RAG?**
   - (Tu respuesta aqui)

3. **Cuales son las limitaciones de usar el LLM como juez para evaluar respuestas?**
   - (Tu respuesta aqui)

4. **Como se comparan estas metricas manuales con herramientas como RAGAS o LangSmith?**
   - (Tu respuesta aqui)

---
## Resumen

En este notebook implementamos tres metricas fundamentales para evaluar sistemas RAG:

1. **Faithfulness (Fidelidad)**: Verifica que la respuesta este respaldada por el contexto,
   descomponiendo la respuesta en afirmaciones y verificando cada una contra el contexto.
   Detecta alucinaciones.

2. **Answer Relevancy (Relevancia)**: Evalua si la respuesta realmente contesta la pregunta
   del usuario, usando el LLM como juez.

3. **Context Precision (Precision del Contexto)**: Mide si el contexto recuperado por el
   retriever es relevante y tiene suficiente cobertura para generar una respuesta correcta.

### Herramientas de evaluacion en produccion
Para sistemas en produccion, considerar:
- **RAGAS**: Framework de evaluacion automatizada para RAG
- **LangSmith**: Plataforma de observabilidad con evaluacion integrada
- **Langfuse**: Herramienta de trazabilidad y evaluacion open source
- **Arize Phoenix**: Observabilidad para modelos de ML e IA

Estas herramientas automatizan y escalan lo que implementamos manualmente en este notebook.